In [4]:
import re
import pandas as pd
import numpy as np
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
filename = "missp.dat.txt"

if os.path.exists(filename):
    print("✅ birkbeck.dat found!")
else:
    print("❌ birkbeck.dat not found!")
    print("Put birkbeck.dat in the same folder as this notebook.")

✅ birkbeck.dat found!


In [6]:
with open(filename, "r", encoding="utf-8", errors="ignore") as file:
    lines = file.readlines()

print("Total lines:", len(lines))

print("\nFirst 30 lines:")
for line in lines[:30]:
    print(line.strip())

Total lines: 42269

First 30 lines:
$Albert
Ab
$America
Ameraca
Amercia
$American
Ameracan
$April
Apirl
$Austrian
Austrain
$Badcock's
badcock
$Bechuanaland
bechuarnia_land
$Botswana
botuania
$Cambridge
Cambrige
$Canada
Canda
$Chautauqua
Chactuquoe
Chalktwa
Chaqua
Chata
Chatacqua
Chatacque
Chatalkwa
Chataqua


In [7]:
def load_vocabulary(filename):
    
    vocabulary = set()

    with open(filename, "r", encoding="utf-8", errors="ignore") as file:
        
        for line in file:
            
            line = line.strip()

            if line.startswith("$"):
                
                word = line[1:].strip().lower()

                if word:
                    vocabulary.add(word)

    return vocabulary


vocabulary = load_vocabulary(filename)

print("Vocabulary size:", len(vocabulary))

Vocabulary size: 6130


In [8]:
print("First 50 vocabulary words:\n")

for word in list(vocabulary)[:50]:
    print(word)

First 50 vocabulary words:

eventuality
experiences
classmates
dose
perch
seen
reverse
futurist
kindest
stuck
exceed
believed
foolish
lighter
advertised
usable
ill
promontory
nicknamed
diagonal
locked
chapel
lived
butcher
break-times
solve
cost
hurry
scatter
woke
rod
corn
short
sitting
does
block
primarily
storm
permissible
leaving
invisible
rejoice
pneumonia
bee
trip
car
fold
chairs
racing
drinking


In [9]:
def tokenize(query):
    return re.findall(r"[a-zA-Z]+", query.lower())


query = "machne lerning cours"

tokens = tokenize(query)

print("Original query:")
print(query)

print("\nTokens:")
print(tokens)

Original query:
machne lerning cours

Tokens:
['machne', 'lerning', 'cours']


In [10]:
def edit_distance(word1, word2):

    m = len(word1)
    n = len(word2)

    dp = np.zeros((m + 1, n + 1), dtype=int)

    # First column
    for i in range(m + 1):
        dp[i][0] = i

    # First row
    for j in range(n + 1):
        dp[0][j] = j

    # Fill DP table
    for i in range(1, m + 1):

        for j in range(1, n + 1):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,       # deletion
                dp[i][j - 1] + 1,       # insertion
                dp[i - 1][j - 1] + cost # replacement
            )

    return dp[m][n]

In [11]:
word1 = "machne"
word2 = "machine"

distance = edit_distance(word1, word2)

print("Word 1:", word1)
print("Word 2:", word2)
print("Edit Distance:", distance)

Word 1: machne
Word 2: machine
Edit Distance: 1


In [12]:
def find_closest_word(word, vocabulary):

    best_word = word
    best_distance = float("inf")

    for candidate in vocabulary:

        distance = edit_distance(word, candidate)

        if distance < best_distance:

            best_distance = distance
            best_word = candidate

    return best_word, best_distance

In [13]:
wrong_word = "machne"

suggestion, distance = find_closest_word(
    wrong_word,
    vocabulary
)

print("Incorrect word:", wrong_word)
print("Suggested correction:", suggestion)
print("Edit distance:", distance)

Incorrect word: machne
Suggested correction: mache
Edit distance: 1


In [14]:
def correct_query(query, vocabulary):

    tokens = tokenize(query)

    corrected_tokens = []
    incorrect_words = []
    suggestions = []

    for word in tokens:

        if word in vocabulary:

            corrected_tokens.append(word)

        else:

            incorrect_words.append(word)

            suggestion, distance = find_closest_word(
                word,
                vocabulary
            )

            suggestions.append(
                (word, suggestion, distance)
            )

            corrected_tokens.append(suggestion)

    corrected_query = " ".join(corrected_tokens)

    return incorrect_words, suggestions, corrected_query

In [15]:
query = "machne lerning cours"

incorrect_words, suggestions, corrected_query = correct_query(
    query,
    vocabulary
)

print("Original Query:")
print(query)

print("\nIncorrect Words:")
print(incorrect_words)

print("\nSuggested Corrections:")

for wrong, correct, distance in suggestions:
    print(
        f"{wrong} → {correct} "
        f"(Edit Distance: {distance})"
    )

print("\nFinal Corrected Query:")
print(corrected_query)

Original Query:
machne lerning cours

Incorrect Words:
['machne', 'lerning', 'cours']

Suggested Corrections:
machne → mache (Edit Distance: 1)
lerning → learning (Edit Distance: 1)
cours → court (Edit Distance: 1)

Final Corrected Query:
mache learning court


In [16]:
results = []

for wrong, correct, distance in suggestions:

    results.append({
        "Incorrect Word": wrong,
        "Suggested Correction": correct,
        "Edit Distance": distance
    })

result_df = pd.DataFrame(results)

display(result_df)

,Incorrect Word,Suggested Correction,Edit Distance
0,machne,mache,1
1,lerning,learning,1
2,cours,court,1


In [17]:
test_queries = [
    "machne lerning cours",
    "artifical inteligence",
    "computr scince",
    "pythn progrming",
    "databse managment"
]

for i, query in enumerate(test_queries, 1):

    print("\n" + "=" * 60)
    print("TEST CASE", i)
    print("=" * 60)

    incorrect_words, suggestions, corrected_query = correct_query(
        query,
        vocabulary
    )

    print("\nOriginal Query:")
    print(query)

    print("\nIncorrect Words:")
    print(incorrect_words)

    print("\nSuggested Corrections:")

    for wrong, correct, distance in suggestions:
        print(
            f"{wrong} → {correct} "
            f"(Edit Distance: {distance})"
        )

    print("\nFinal Corrected Query:")
    print(corrected_query)


TEST CASE 1

Original Query:
machne lerning cours

Incorrect Words:
['machne', 'lerning', 'cours']

Suggested Corrections:
machne → mache (Edit Distance: 1)
lerning → learning (Edit Distance: 1)
cours → court (Edit Distance: 1)

Final Corrected Query:
mache learning court

TEST CASE 2

Original Query:
artifical inteligence

Incorrect Words:
['artifical', 'inteligence']

Suggested Corrections:
artifical → artificial (Edit Distance: 1)
inteligence → intelligence (Edit Distance: 1)

Final Corrected Query:
artificial intelligence

TEST CASE 3

Original Query:
computr scince

Incorrect Words:
['computr', 'scince']

Suggested Corrections:
computr → compass (Edit Distance: 3)
scince → science (Edit Distance: 1)

Final Corrected Query:
compass science

TEST CASE 4

Original Query:
pythn progrming

Incorrect Words:
['pythn', 'progrming']

Suggested Corrections:
pythn → path (Edit Distance: 2)
progrming → progressing (Edit Distance: 3)

Final Corrected Query:
path progressing

TEST CASE 5

Orig

In [18]:
query = input("Enter your search query: ")

incorrect_words, suggestions, corrected_query = correct_query(
    query,
    vocabulary
)

print("\n" + "=" * 60)
print("SEARCH QUERY SPELLING CORRECTOR")
print("=" * 60)

print("\nOriginal Query:")
print(query)

print("\nIncorrect Words:")

if incorrect_words:
    for word in incorrect_words:
        print("-", word)
else:
    print("No spelling errors found.")

print("\nSuggested Corrections:")

if suggestions:
    for wrong, correct, distance in suggestions:
        print(
            f"{wrong} → {correct} "
            f"(Edit Distance: {distance})"
        )
else:
    print("No corrections required.")

print("\nFinal Corrected Query:")
print(corrected_query)

print("=" * 60)

Enter your search query:  artficial



SEARCH QUERY SPELLING CORRECTOR

Original Query:
artficial

Incorrect Words:
- artficial

Suggested Corrections:
artficial → artificial (Edit Distance: 1)

Final Corrected Query:
artificial
